# Part 3: PyTorch and Deep Learning

Prerequisites: Parts 1 and 2, especially Section 11 of Part 1 (manual gradient descent and MLPs). Here we implement the same ideas in PyTorch: tensors, automatic gradients, `nn.Module`, training loops, and CNNs.

**Installation** (a CPU is sufficient for these examples; for GPU support use the command from pytorch.org appropriate for your CUDA version):
```bash
pip install torch torchvision scikit-learn matplotlib pandas
```

## Contents
1. Tensors and autograd
2. Linear regression: three levels
3. The training loop: Dataset, DataLoader, train/val (MLP, dropout, batchnorm, weight decay, early stopping)
4. Optimizers and learning rate (SGD, momentum, Adam, scheduler)
5. CNNs: neural networks for images (digits)
6. Checklist: "the network is not learning"
7. Saving models and GPU usage
8. Next steps in DL: transfer learning, Transformers, NLP/CV


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, time, copy, warnings
warnings.filterwarnings("ignore")
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset

torch.manual_seed(0); np.random.seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device)

## 1. Tensors and autograd

A **tensor** resembles a NumPy array with two additional capabilities: GPU computation and **automatic gradient calculation**.

`requires_grad=True` tells PyTorch to record operations in a **computation graph**. `loss.backward()` traverses the graph backward (backpropagation) and stores parameter gradients in `.grad`.


In [ ]:
x = torch.tensor([[1., 2.], [3., 4.]])
print(x.shape, x.dtype, x.device)
print(x @ x, "\n", x.mean(), x.sum(dim=0), x.T)
print(torch.from_numpy(np.arange(6).reshape(2, 3)).float())   # numpy <-> torch

# autograd: f(w) = (w1*2 + w2*3 - 5)^2, gradient with respect to w
w = torch.tensor([1.0, 1.0], requires_grad=True)
f = (w[0] * 2 + w[1] * 3 - 5) ** 2
f.backward()
print("f =", f.item(), "| df/dw =", w.grad)      # manually: 2*(2+3-5)*[2,3] = [0,0]

w = torch.tensor([2.0, 3.0], requires_grad=True)
f = (w[0] * 2 + w[1] * 3 - 5) ** 2
f.backward()
print("df/dw =", w.grad, "(verify manually: 2*(4+9-5)*[2,3] = [32, 48])")

Important rules:
- Gradients **accumulate**: call `optimizer.zero_grad()` before each update.
- `with torch.no_grad():` disables graph recording for inference and validation, reducing overhead and memory use.
- `model.train()` and `model.eval()` switch Dropout/BatchNorm behavior. **Forgetting `eval()` during validation is a common bug.**


### 📝 Exercise 1.1
Create a tensor `w` with requires_grad and calculate `L = ((w*x - y)**2).mean()` for x=[1,2,3], y=[2,4,6], w=0.5. Call `backward()` and verify `w.grad` manually using dL/dw = mean(2*x*(w*x - y)).


In [ ]:
# Your solution here

### 📝 Exercise 1.2
Perform 20 manual gradient descent steps for w (`with torch.no_grad(): w -= lr * w.grad; w.grad.zero_()`). What value does w approach?


In [ ]:
# Your solution here

## 2. Linear regression: three levels

Reach the same result in three ways to understand what `nn.Module` does internally.


In [ ]:
# data: y = 3x + 2 + noise
Xr = torch.linspace(-3, 3, 200).unsqueeze(1)
yr = 3 * Xr + 2 + 0.5 * torch.randn_like(Xr)

# Level 1: parameters + autograd, manual updates
w = torch.zeros(1, 1, requires_grad=True); b_ = torch.zeros(1, requires_grad=True)
for ep in range(300):
    pred = Xr @ w + b_
    loss = F.mse_loss(pred, yr)
    loss.backward()
    with torch.no_grad():
        w -= 0.05 * w.grad; b_ -= 0.05 * b_.grad
        w.grad.zero_(); b_.grad.zero_()
print("level 1:", w.item(), b_.item())

# Level 2: nn.Linear + optimizer
lin = nn.Linear(1, 1)
opt = torch.optim.SGD(lin.parameters(), lr=0.05)
for ep in range(300):
    opt.zero_grad()
    loss = F.mse_loss(lin(Xr), yr)
    loss.backward()
    opt.step()
print("level 2:", lin.weight.item(), lin.bias.item())

# Level 3: custom nn.Module (the standard network pattern)
class LinReg(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(1, 1)
    def forward(self, x):
        return self.fc(x)

m = LinReg(); opt = torch.optim.Adam(m.parameters(), lr=0.1)
for ep in range(300):
    opt.zero_grad(); F.mse_loss(m(Xr), yr).backward(); opt.step()
print("level 3:", m.fc.weight.item(), m.fc.bias.item(), "(expected: approximately 3 and 2)")

### 📝 Exercise 2.1
Fit the nonlinear relationship y = sin(x) + noise using `nn.Sequential(nn.Linear(1,32), nn.ReLU(), nn.Linear(32,32), nn.ReLU(), nn.Linear(32,1))`. Plot the data and predictions. Try `nn.Tanh()` instead of ReLU, then remove nonlinearities entirely. What happens?


In [ ]:
# Your solution here

## 3. The training loop: Dataset, DataLoader, train/val

A typical PyTorch project has this structure:

```python
for epoch in range(E):
    model.train()
    for xb, yb in train_loader:      # mini-batch
        optimizer.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        optimizer.step()
    model.eval()
    with torch.no_grad():
        ...  # evaluate on val
```

- **Dataset** retrieves one example `(x, y)`.
- **DataLoader** groups examples into batches, shuffles them, and can parallelize loading.
- Validation is **essential**, just as in sklearn: without it, overfitting is difficult to detect.


In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

Xb, yb_ = load_breast_cancer(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(Xb, yb_, test_size=0.2, stratify=yb_, random_state=0)
Xtr, Xva, ytr, yva = train_test_split(Xtr, ytr, test_size=0.2, stratify=ytr, random_state=0)

sc = StandardScaler().fit(Xtr)                      # fit scaler only on train to avoid data leakage
def T(a, dtype=torch.float32): return torch.tensor(a, dtype=dtype)
Xtr_t, Xva_t, Xte_t = T(sc.transform(Xtr)), T(sc.transform(Xva)), T(sc.transform(Xte))
ytr_t, yva_t, yte_t = T(ytr).unsqueeze(1), T(yva).unsqueeze(1), T(yte).unsqueeze(1)

train_loader = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=32, shuffle=True)
print(len(train_loader), "batches;", Xtr_t.shape)

In [ ]:
class MLP(nn.Module):
    def __init__(self, d_in, hidden=(64, 32), p_drop=0.0, batchnorm=False):
        super().__init__()
        layers, prev = [], d_in
        for h in hidden:
            layers.append(nn.Linear(prev, h))
            if batchnorm: layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            if p_drop > 0: layers.append(nn.Dropout(p_drop))
            prev = h
        layers.append(nn.Linear(prev, 1))            # logit (no sigmoid: BCEWithLogitsLoss includes it)
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

def evaluate(model, X, y, loss_fn):
    model.eval()
    with torch.no_grad():
        logits = model(X)
        loss = loss_fn(logits, y).item()
        prob = torch.sigmoid(logits).numpy().ravel()
    return loss, roc_auc_score(y.numpy().ravel(), prob)

def fit(model, train_loader, Xva, yva, epochs=100, lr=1e-3, weight_decay=0.0, patience=15, verbose=True):
    loss_fn = nn.BCEWithLogitsLoss()
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    hist = {"train": [], "val": [], "val_auc": []}
    best, best_state, bad = float("inf"), None, 0
    for ep in range(epochs):
        model.train(); run = 0.0; n = 0
        for xb, yb in train_loader:
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward(); opt.step()
            run += loss.item() * len(xb); n += len(xb)
        vloss, vauc = evaluate(model, Xva, yva, loss_fn)
        hist["train"].append(run / n); hist["val"].append(vloss); hist["val_auc"].append(vauc)
        if vloss < best - 1e-4:
            best, best_state, bad = vloss, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= patience:
                if verbose: print(f"early stop at epoch {ep}")
                break
    model.load_state_dict(best_state)                # restore the best validation state
    return hist

torch.manual_seed(0)
model = MLP(Xtr_t.shape[1], (64, 32), p_drop=0.2)
hist = fit(model, train_loader, Xva_t, yva_t, epochs=200, lr=1e-3, weight_decay=1e-4)

plt.plot(hist["train"], label="train loss"); plt.plot(hist["val"], label="val loss")
plt.xlabel("epoch"); plt.legend(); plt.title("Learning curves"); plt.show()

tl, ta = evaluate(model, Xte_t, yte_t, nn.BCEWithLogitsLoss())
print(f"TEST loss={tl:.4f}  AUC={ta:.4f}")

**Reading learning curves to diagnose a DL model:**
- Both train and val loss remain high → underfitting (increase model capacity/lr, train longer, check the data)
- Train loss falls while val loss rises → overfitting (dropout, weight decay, a smaller model, more data, early stopping, augmentation)
- Val loss fluctuates strongly → lr may be too large or the batch too small
- Loss = `nan` → excessive lr, division by zero, log of zero, or invalid data


### 📝 Exercise 3.1
Train three MLP variants: (a) no regularization, (b) `p_drop=0.3`, (c) `weight_decay=1e-2`, all with `patience=1000` (no early stopping) for 300 epochs. Plot their validation losses together. Which overfits the least?


In [ ]:
# Your solution here

### 📝 Exercise 3.2
Compare this MLP with sklearn's `LogisticRegression` and `HistGradientBoostingClassifier` using test AUC. Which performs best? What does this suggest about neural networks on small tabular datasets?


In [ ]:
# Your solution here

### 📝 Exercise 3.3
Implement a custom `Dataset` (a class with `__len__` and `__getitem__`) in place of `TensorDataset` and verify that it produces the same result.


In [ ]:
# Your solution here

## 4. Optimizers and learning rate

- **SGD**: `w ← w − lr·g`. Simple, but sensitive to lr.
- **SGD + momentum**: accumulates "velocity", smoothing noise and accelerating progress along valleys.
- **Adam**: adaptive per-parameter learning rates + momentum. A useful default (`lr=1e-3`). **AdamW**, with decoupled weight decay, is often a good alternative.
- **LR scheduler**: adjusts lr during training (`StepLR`, `CosineAnnealingLR`, `ReduceLROnPlateau`, `OneCycleLR`).
- Learning rate is a **key hyperparameter** in DL. Explore orders of magnitude: 1e-4, 1e-3, 1e-2.


In [ ]:
def run(opt_name, lr, epochs=60, seed=0):
    torch.manual_seed(seed)
    m = MLP(Xtr_t.shape[1], (64, 32))
    opt = {"sgd": lambda p: torch.optim.SGD(p, lr=lr),
           "momentum": lambda p: torch.optim.SGD(p, lr=lr, momentum=0.9),
           "adam": lambda p: torch.optim.Adam(p, lr=lr),
           "adamw": lambda p: torch.optim.AdamW(p, lr=lr, weight_decay=1e-2)}[opt_name](m.parameters())
    loss_fn = nn.BCEWithLogitsLoss(); curve = []
    for ep in range(epochs):
        m.train()
        for xb, yb in train_loader:
            opt.zero_grad(); loss_fn(m(xb), yb).backward(); opt.step()
        curve.append(evaluate(m, Xva_t, yva_t, loss_fn)[0])
    return curve

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for n_, lr in [("sgd", 0.01), ("momentum", 0.01), ("adam", 0.001), ("adamw", 0.001)]:
    ax[0].plot(run(n_, lr), label=f"{n_} lr={lr}")
ax[0].set_title("Optimizers"); ax[0].legend(); ax[0].set_ylabel("val loss")
for lr in [1e-4, 1e-3, 1e-2, 1e-1, 1.0]:
    ax[1].plot(run("adam", lr), label=f"lr={lr}")
ax[1].set_title("Adam: effect of lr"); ax[1].legend(); ax[1].set_ylim(0, 1.5)
plt.show()

### 📝 Exercise 4.1
Add `torch.optim.lr_scheduler.CosineAnnealingLR` to `fit` (call `scheduler.step()` once per epoch). Does it improve performance or accelerate convergence?


In [ ]:
# Your solution here

### 📝 Exercise 4.2
BatchNorm experiment: compare `batchnorm=False/True` at lr=0.1 with Adam. BatchNorm often makes a network more tolerant of larger learning rates. Does the experiment support this?


In [ ]:
# Your solution here

## 5. CNNs: neural networks for images

A fully connected network does not explicitly account for spatial relationships between neighboring pixels and uses many parameters. **Convolutional layers** address this:

- **Conv2d**: a small filter (e.g., 3×3) slides across the image, detecting local patterns such as edges, corners, and textures. **Weights are shared** across the image (fewer parameters, translation equivariance).
- **Pooling** (`MaxPool2d`): reduces spatial size while retaining the strongest response, helping robustness to shifts.
- Deeper layers combine simple features into complex ones (edges → shapes → objects).
- Tensor format: `(batch, channels, height, width)`.

We use sklearn's `load_digits` (8×8 digit images; no download required).


In [ ]:
from sklearn.datasets import load_digits
dg = load_digits()
Xd = dg.images.astype(np.float32) / 16.0                # (1797, 8, 8) in the range [0,1]
yd = dg.target
fig, ax = plt.subplots(1, 8, figsize=(14, 2))
for i, a_ in enumerate(ax): a_.imshow(Xd[i], cmap="gray"); a_.set_title(yd[i]); a_.axis("off")
plt.show()

Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(Xd, yd, test_size=0.2, stratify=yd, random_state=0)
Xd_tr, Xd_va, yd_tr, yd_va = train_test_split(Xd_tr, yd_tr, test_size=0.2, stratify=yd_tr, random_state=0)
img = lambda a: torch.tensor(a).unsqueeze(1)             # (N,1,8,8)
lab = lambda a: torch.tensor(a, dtype=torch.long)
dtr = DataLoader(TensorDataset(img(Xd_tr), lab(yd_tr)), batch_size=64, shuffle=True)
Xv, yv = img(Xd_va), lab(yd_va); Xt, yt = img(Xd_te), lab(yd_te)
print(img(Xd_tr).shape)

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(),       # (16,8,8)
            nn.MaxPool2d(2),                                  # (16,4,4)
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(),      # (32,4,4)
            nn.MaxPool2d(2),                                  # (32,2,2)
        )
        self.head = nn.Sequential(nn.Flatten(), nn.Dropout(0.3), nn.Linear(32 * 2 * 2, 64), nn.ReLU(), nn.Linear(64, n_classes))
    def forward(self, x): return self.head(self.features(x))

class FlatMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Flatten(), nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, 10))
    def forward(self, x): return self.net(x)

def train_clf(model, epochs=40, lr=2e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr); lf = nn.CrossEntropyLoss(); hist = []
    for ep in range(epochs):
        model.train()
        for xb, yb in dtr:
            opt.zero_grad(); lf(model(xb), yb).backward(); opt.step()
        model.eval()
        with torch.no_grad():
            hist.append((model(Xv).argmax(1) == yv).float().mean().item())
    return hist

torch.manual_seed(0)
cnn, mlp = SmallCNN(), FlatMLP()
h_cnn, h_mlp = train_clf(cnn), train_clf(mlp)
print("parameters: CNN", sum(p.numel() for p in cnn.parameters()), "| MLP", sum(p.numel() for p in mlp.parameters()))
plt.plot(h_cnn, label="CNN"); plt.plot(h_mlp, label="MLP"); plt.legend(); plt.ylabel("val accuracy"); plt.show()

cnn.eval()
with torch.no_grad(): print("CNN test acc:", (cnn(Xt).argmax(1) == yt).float().mean().item())

Keep these output/loss combinations in mind:
- **Multiclass classification**: `CrossEntropyLoss`, raw output logits (no softmax in the model), integer class labels
- **Binary classification**: `BCEWithLogitsLoss`, one logit per example, float 0/1 labels
- **Regression**: `MSELoss`, `L1Loss`, or `HuberLoss`


### 📝 Exercise 5.1
Explore how CNNs handle **position**: shift all test images by 1 pixel using `torch.roll` and compare the accuracy drop for CNN and FlatMLP. Why might the CNN be more robust?


In [ ]:
# Your solution here

### 📝 Exercise 5.2
Implement **augmentation**: randomly shift each image by ±1 pixel during training. Does test accuracy improve?


In [ ]:
# Your solution here

### 📝 Exercise 5.3
Visualize the learned filters in the first convolutional layer (`cnn.features[0].weight`, shape (16,1,3,3)) as 16 small images. What patterns do they detect?


In [ ]:
# Your solution here

### 📝 Exercise 5.4
Build a confusion matrix for the CNN on test using `sklearn.metrics`. Which digits are most frequently confused?


In [ ]:
# Your solution here

## 6. Checklist: "the network is not learning"

1. **Overfit 10–20 examples.** If a model cannot memorize a tiny dataset to loss≈0, investigate the implementation before scaling up. This is a useful first check in a project.
2. Check **tensor shapes** at each step (`print(x.shape)`).
3. Check the **initial loss**: for 10 classes, expect roughly `ln(10) ≈ 2.3`. Investigate large deviations.
4. Decrease **lr** by a factor of 10 (or increase it). This helps diagnose convergence issues.
5. Are `zero_grad()`, `model.train()/eval()`, and `no_grad()` called in the appropriate places?
6. Are the inputs **scaled**? Normalization matters for neural networks.
7. Do the **loss and outputs** match (logits vs probabilities; applying softmax/sigmoid twice is a common bug)?
8. Is there **leakage** (a scaler fitted on all data, duplicates across train/val)?
9. Inspect **misclassified examples**: they often reveal data problems.
10. Fix the random **seed** and compare one change at a time.


In [ ]:
# Check 1: overfit a tiny dataset. Investigate if loss does not approach 0.
torch.manual_seed(0)
tiny_x, tiny_y = img(Xd_tr[:16]), lab(yd_tr[:16])
m = SmallCNN(); opt = torch.optim.Adam(m.parameters(), lr=3e-3); lf = nn.CrossEntropyLoss()
print("initial loss:", lf(m(tiny_x), tiny_y).item(), "(expected: approximately ln(10)=2.3)")
for i in range(200):
    m.train(); opt.zero_grad(); l = lf(m(tiny_x), tiny_y); l.backward(); opt.step()
print("after 200 steps:", l.item())

## 7. Saving models and GPU usage

- Save the **`state_dict`** rather than the entire object: `torch.save(model.state_dict(), "m.pt")`.
- To load: create the same model class, call `model.load_state_dict(torch.load("m.pt"))`, then `model.eval()`.
- GPU: use `model.to(device)` and move **every batch** with `xb.to(device)`, `yb.to(device)`. Model and data must be on the same device.


In [ ]:
torch.save(cnn.state_dict(), "digits_cnn.pt")
restored = SmallCNN(); restored.load_state_dict(torch.load("digits_cnn.pt")); restored.eval()
with torch.no_grad(): print("restored model, acc:", (restored(Xt).argmax(1) == yt).float().mean().item())

# GPU template (also works on CPU)
model_g = SmallCNN().to(device)
xb, yb = next(iter(dtr))
out = model_g(xb.to(device))
print("output device:", out.device, out.shape)

### 📝 Exercise 7.1
Wrap training in a function that uses `device` and measure the time for 20 epochs on CPU and, if available, CUDA. When is a GPU faster, and when is it not? Try small and large batches.


In [ ]:
# Your solution here

## 8. Next steps in DL

### Step 1: Real-world images (1–2 weeks)
- **Fashion-MNIST / CIFAR-10** through `torchvision.datasets`.
- A larger CNN, BatchNorm, augmentation (`torchvision.transforms`), and a scheduler.
- **Transfer learning**: use `resnet18` from `torchvision.models`, pretrained on ImageNet; replace its final layer for the task and fine-tune it. This is a common practical computer-vision workflow. Project: an image classifier using an open dataset or educational photo collection, such as types of electronic devices.

### Step 2: Text and sequences (2–3 weeks)
- Tokenization, embeddings, RNN/LSTM for background, then **Transformers**.
- **Attention**: a central idea in modern AI. Study "Attention Is All You Need" alongside a manual self-attention implementation (~30 lines of PyTorch).
- Practice with **Hugging Face Transformers**: fine-tune `distilbert` for review classification.

### Step 3: Deeper understanding
- Manual backpropagation (Karpathy's "micrograd"), then "makemore" and "nanoGPT". The *Neural Networks: Zero to Hero* series develops these ideas in detail.
- Books: *Deep Learning with PyTorch* or *Dive into Deep Learning* (d2l.ai, free, with PyTorch code).
- fast.ai Practical Deep Learning: a top-down approach emphasizing practical results.

### Step 4: From notebook to product (MLOps essentials)
- Inference via **FastAPI / Django REST**, containerization (Docker), experiment tracking (**MLflow / Weights & Biases**), and drift monitoring.
- Integrating an ML model into a web application combines data preparation, inference, and API development. Prior experience with a particular web framework is not required.

### Suggested one-to-two-month plan
| Week | Focus |
|---|---|
| 1 | Complete Parts 1 and 2, projects 1–2 |
| 2 | Part 3: Sections 1–4, custom MLP on tabular data |
| 3 | CNNs, Fashion-MNIST/CIFAR, transfer learning |
| 4 | Transfer-learning project or ML functionality in a web application |
| 5–6 | Text: embeddings, attention, fine-tuning distilbert |
| 7–8 | Karpathy's Zero to Hero, deployment, portfolio |

### Completion checklist
- [ ] Write an `nn.Module` and training loop independently
- [ ] Explain `zero_grad`, `backward`, `step`, `train/eval`, and `no_grad`
- [ ] Interpret learning curves and draw useful conclusions
- [ ] Know when DL is appropriate (images, text, audio) and when boosting may work better (tabular data)
- [ ] Perform transfer learning
- [ ] Understand the idea of attention
